In [1]:
import pandas as pd
import numpy as np
import os

columns = [
    'Time', 'attitude_roll', 'attitude_pitch', 'attitude_yaw',
    'rotation_rate_x', 'rotation_rate_y', 'rotation_rate_z',
    'gravity_x', 'gravity_y', 'gravity_z',
    'user_acc_x', 'user_acc_y', 'user_acc_z',
    'magnetic_field_x', 'magnetic_field_y', 'magnetic_field_z'
]

def load_imu_data(file_path):
    try:
        data = pd.read_csv(file_path, sep=",", header=None, names=columns)  
        return data
    except Exception as e:
        print(f"Error loading data from file: {e}")
        return None

def extract_and_combine_data(data):
    if data is not None:
        # Extract relevant columns (accel, magnetic, rotation rate)
        acc_data = data[['user_acc_x', 'user_acc_y', 'user_acc_z']].values
        rotation_rate_data = data[['rotation_rate_x', 'rotation_rate_y', 'rotation_rate_z']].values
        magnetic_data = data[['magnetic_field_x', 'magnetic_field_y', 'magnetic_field_z']].values
        
        #normalize
        acc_mean, acc_std = np.mean(acc_data), np.std(acc_data)
        gyro_mean, gyro_std = np.mean(rotation_rate_data), np.std(rotation_rate_data)
        mag_mean, mag_std = np.mean(magnetic_data), np.std(magnetic_data)

        # Standardize the data
        acc_data = (acc_data - acc_mean) / acc_std
        rotation_rate_data = (rotation_rate_data - gyro_mean) / gyro_std
        magnetic_data = (magnetic_data - mag_mean) / mag_std
        
        combined_data = np.hstack((acc_data, rotation_rate_data, magnetic_data))
        print(combined_data.shape)        
        return combined_data
    else:
        return None
    
def segmentData(accData,time_step,step):
    step = int(step)
    segmentAccData = []
    for i in range(0, accData.shape[0] - time_step,step):
        segmentAccData.append(accData[i:i+time_step,:])
    return np.asarray(segmentAccData)

In [2]:
Dataset_path = "dataset/ours/train"
file_path = "Sensor/imu.txt"
label_path = "label.txt"

all_items = os.listdir(Dataset_path)
folders = [item for item in all_items if os.path.isdir(os.path.join(Dataset_path, item))]

client_data = {iniArray: [] for iniArray in range(16)}
client_label = {iniArray: [] for iniArray in range(16)}
for index,folder_path in enumerate(folders):
    print("index",index)
    full_path = os.path.join(Dataset_path, folder_path, file_path)
    imu_data = load_imu_data(full_path)
    print(f"Processing {full_path}")

    imu_data_processed = extract_and_combine_data(imu_data)
    if imu_data_processed is not None:
        print(f"Processed IMU data shape: {imu_data_processed.shape}")
        
        segmented_data = segmentData(imu_data_processed, time_step=128, step=64)
        print(f"Segmented data shape: {segmented_data.shape}")
        
        client_data[index] = segmented_data
        
        label_file_path = os.path.join(Dataset_path, folder_path, label_path)
        if os.path.exists(label_file_path):
            print(f"Found label.txt at: {label_file_path}")
            with open(label_file_path, 'r') as file:
                label_data = file.read().strip()
            client_label[index].append(np.full(segmented_data.shape[0], label_data, dtype=int))
        else:
            print(f"Label file not found for {folder_path}")
        

index 0
Processing dataset/ours/train/DEFAULT_DEFAULT_2025-01-03_19-02-28/Sensor/imu.txt
(9959, 9)
Processed IMU data shape: (9959, 9)
Segmented data shape: (154, 128, 9)
Found label.txt at: dataset/ours/train/DEFAULT_DEFAULT_2025-01-03_19-02-28/label.txt
index 1
Processing dataset/ours/train/DEFAULT_DEFAULT_2025-01-03_19-13-34/Sensor/imu.txt
(10458, 9)
Processed IMU data shape: (10458, 9)
Segmented data shape: (162, 128, 9)
Found label.txt at: dataset/ours/train/DEFAULT_DEFAULT_2025-01-03_19-13-34/label.txt
index 2
Processing dataset/ours/train/DEFAULT_DEFAULT_2025-01-03_19-22-46/Sensor/imu.txt
(10365, 9)
Processed IMU data shape: (10365, 9)
Segmented data shape: (160, 128, 9)
Found label.txt at: dataset/ours/train/DEFAULT_DEFAULT_2025-01-03_19-22-46/label.txt
index 3
Error loading data from file: Error tokenizing data. C error: Expected 16 fields in line 5013, saw 31

Processing dataset/ours/train/DEFAULT_DEFAULT_2025-01-03_19-34-47/Sensor/imu.txt
index 4
Processing dataset/ours/trai

In [3]:
processedData = []
processedLabel = []
clientSize = []
for clientIndex in range(16):
    if len(client_data[clientIndex]) == 0:
        print(f"No data for client {clientIndex}")
        continue
    processedData.append(client_data[clientIndex])
    processedLabel.append(client_label[clientIndex][0])

No data for client 3


In [4]:
# print(processedData[0])
print(len(processedData))
print(processedData[4].shape)
# print(processedLabel[4])
# print(processedLabel[4].shape)
combinedUserData = np.vstack((processedData))
print(combinedUserData.shape)

15
(38, 128, 9)
(1461, 128, 9)


In [6]:
import hickle as hkl


startIndex = 0
endIndex = 0 
dataName = 'Ours'
os.makedirs('datasetStandardized_s3/'+dataName, exist_ok=True)

for i in range(len(processedData)):
    startIndex = endIndex 
    endIndex = startIndex +  processedData[i].shape[0]
    hkl.dump(combinedUserData[startIndex:endIndex],'datasetStandardized_s3/'+dataName+'/UserData'+str(i)+'.hkl' )
    hkl.dump(processedLabel[i],'datasetStandardized_s3/'+dataName+'/UserLabel'+str(i)+'.hkl' )